# 03. SARIMAX 분석 (N / A / B / C)

**이 노트북이 하는 일**

역별 `log(total)`을 SARIMAX로 모델링해 날씨·특보의 효과와 예측 성능을 비교합니다. 기존 `scripts/sarimax_asos.py`의 stage 5 로직을 그대로 옮겼고, 맨 아래에는 **`summer_vacation` 있음/없음 B 모형 비교(stage 3)**를 현재 설정(근로자의 날·명절 ±2일·`alert_unknown`을 포함한 `MASK_COLS`)으로 다시 돌립니다 (그림 7의 입력).

- **N**: 날씨 없음 (계단 더미, summer_vacation만)
- **A**: 특보만 (특보 4개와 lag1~3)
- **B**: ASOS 날씨만 (당일 + lag1~3, 사후 상한선)  /  **B_lag**: 당일을 뺀 lag1~3만 (예측 시점에 아는 정보)
- **C**: 특보 + ASOS  /  **C_lag**: 특보 + ASOS lag만

설정은 모두 `00_settings`에 있습니다: 차수 (1,0,1)(1,1,1,7), 2014~2023 학습, 2024 롤링 예측(계수 고정, 하루 앞), 가리기 열 `MASK_COLS`가 1인 날은 y=NaN. **2025년 행은 읽자마자 버리며 쓰지 않습니다.** 수유·연신내의 C는 B 계수를 초기값으로 재시도합니다.

**입력 파일**

- `data/processed/model_input_9stations.csv`

**출력 파일**

- `results/sarimax_NABC_compare.csv` (역×모형 지표), `sarimax_NABC_by_type.csv` (유형별 요약), `sarimax_NABC_vs_N.csv` (N 대비 개선·Wilcoxon p), `sarimax_NABC_pred2024.csv` (일별 예측), `sarimax_NABC_coefs.csv`, `sarimax_NABC_cumulative.csv`, `sarimax_B_season_compare.csv`, `sarimax_B_season_coefs.csv`, `sarimax_B_season_cumulative.csv`, `sarimax_B_season_by_type.csv` (summer_vacation 있음/없음 비교), `sarimax_NABC_alert_significance.csv`


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다. 병렬 적합에는 `joblib`을 씁니다 (노트북 안에서 정의한 함수도 병렬로 돌릴 수 있음).

In [2]:
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

**입력 파일 경로**: 모델 입력 표.

In [3]:
DATA = PROC / 'model_input_9stations.csv'

**`label`**: 차수를 `(1,0,1)(1,1,1,7)` 같은 글자로 만듭니다.

In [4]:
def label(order, seasonal):
    return f'{order}{seasonal}'.replace(' ', '')

확정 차수를 글자로 만든 이름 (결과 제목에 씀).

In [5]:
MAIN = label(*CANDIDATES[0])

**`load`**: `model_input_9stations.csv`를 읽고 2025(test) 행은 바로 버립니다.

In [6]:
def load():
    d = pd.read_csv(DATA, parse_dates=['date'])
    return d[d.date <= VALID_END]                              # 2025(test)는 여기서 버림

**`prepare`**: 한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.

In [7]:
def prepare(d, station, season=False, model='B'):
    """한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.
    model: 'B' = ASOS 실측만, 'A' = 특보만, 'C' = 특보 + ASOS, 'N' = 날씨 없음, 'B_lag'/'C_lag' = B/C에서 당일 ASOS를 뺀 버전"""
    s = d[d.station == station].set_index('date').sort_index().asfreq('D')
    assert s.index.min() == START and s.index.max() == VALID_END and s.station.notna().all(), '날짜 빈틈'
    y = np.log(s.total)
    y[s[MASK_COLS].sum(axis=1) > 0] = np.nan                 # 가리기: 날짜는 유지, y만 NaN
    blocks = []
    if model in ('B', 'C', 'B_lag', 'C_lag'):
        w = pd.DataFrame({
            'heat_excess': (s.feels_like_max - HEAT_BASE).clip(lower=0).fillna(0),
            'cold_excess': (COLD_BASE - s.tmin).clip(lower=0).fillna(0),
            'rain': np.log1p(s.rain_mm),
            'snow': s.snow_new_cm,
        }).fillna(0)
        blocks += ([] if model.endswith('_lag') else [w]) + [pd.DataFrame({f'{c}_lag{k}': w[c].shift(k).fillna(0) for c in WEATHER for k in LAGS})]
    if model in ('A', 'C', 'C_lag'):
        a = s[ALERTS].fillna(0)                                 # NaN은 2014-01-01~07뿐(y도 가려짐)
        blocks += [a, pd.DataFrame({f'{c}_lag{k}': a[c].shift(k).fillna(0) for c in ALERTS for k in LAGS})]
    steps = list(STEP_COLS) + (list(SEASON_COLS) if season else []) + ([STATION_STEP[station]] if station in STATION_STEP else [])
    X = pd.concat([s[steps]] + blocks, axis=1).astype(float)
    return y, X, s.group.iloc[0], s.total

**`_fit`**: SARIMAX를 한 번 적합합니다(경고를 모아 둠).

In [8]:
def _fit(model, start=None, maxiter=None, method=None, **kw):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        res = model.fit(start_params=start, disp=False, method=method or FIT_METHOD, maxiter=maxiter or FIT_MAXITER, **kw)
    return res, sorted({f'{w.category.__name__}: {str(w.message)[:110]}' for w in caught})

**`_converged`**: 적합이 수렴했는지 확인합니다.

In [9]:
def _converged(res):
    return bool(res.mle_retvals.get('converged'))

**`alt_start`**: 다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값

In [10]:
def alt_start(model, ytr, Xtr):
    """다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값"""
    dy, dX = ytr - ytr.shift(7), Xtr - Xtr.shift(7)
    ok = dy.notna() & dX.notna().all(axis=1)
    beta = np.linalg.lstsq(dX[ok].values, dy[ok].values, rcond=None)[0]
    sig2 = float(np.var(dy[ok].values - dX[ok].values @ beta))
    cols, out = list(Xtr.columns), []
    for n in model.param_names:
        if n in cols: out.append(beta[cols.index(n)])
        elif n.startswith('ar.S'): out.append(0.1)
        elif n.startswith('ma.S'): out.append(-0.8)
        elif n == 'ar.L1': out.append(0.5)
        elif n.startswith(('ar.', 'ma.')): out.append(0.0)
        else: out.append(sig2)
    return np.array(out)

**`fit_station`**: 2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.

In [11]:
def fit_station(y, X, order, seasonal_order, retry=False, hint=None):
    """2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.
    retry=True면 기본 설정(lbfgs, maxiter=1000)이 수렴하지 않을 때 순서대로 다시 시도:
    (1) 기본 결과의 계수에서 이어서 maxiter=5000, (2) 다른 초기값에서 maxiter=5000. 첫 수렴 결과를 쓰고, 없으면 loglike가 가장 큰 것을 씀.
    hint(다른 모형의 계수 {이름: 값})를 주면 그 계수를 초기값으로 lbfgs(maxiter=5000)를 먼저 돌리고, 수렴하지 않으면
    그 초기값에서 method='nm'을 먼저 돌린 뒤 그 결과로 lbfgs(maxiter=5000)를 다시 돌린다(없는 변수는 0). 그래도 안 되면 미수렴 표시"""
    ytr, Xtr = y[:TRAIN_END], X[:TRAIN_END]
    dropped = [c for c in X.columns if (Xtr[c] == 0).all()]
    X = X.drop(columns=dropped); Xtr = Xtr.drop(columns=dropped)
    model = SARIMAX(ytr, exog=Xtr, order=order, seasonal_order=seasonal_order)
    if hint is not None:
        start = np.array([hint.get(n, 0.0) for n in model.param_names])
        r1, m1 = _fit(model, start=start, maxiter=5000)
        tried = [('B 계수를 초기값으로(lbfgs, maxiter=5000)', r1, m1)]
        if not _converged(r1):
            rn, _ = _fit(model, start=start, method='nm', maxiter=3000, maxfun=6000)
            r2, m2 = _fit(model, start=rn.params.values, maxiter=5000)
            tried.append(('B 계수 초기값, nm 후 lbfgs(maxiter=5000)', r2, m2))
    else:
        res, msgs = _fit(model)
        tried = [('기본(lbfgs, maxiter=1000)', res, msgs)]
        if retry and not _converged(res):
            r1, m1 = _fit(model, start=res.params.values, maxiter=5000)
            tried.append(('기본 결과에서 이어서(maxiter=5000)', r1, m1))
            if not _converged(r1):
                r2, m2 = _fit(model, start=alt_start(model, ytr, Xtr), maxiter=5000)
                tried.append(('다른 초기값(차분 회귀, maxiter=5000)', r2, m2))
    best = next((t for t in tried if _converged(t[1])), None) or max(tried, key=lambda t: t[1].llf)
    note = best[0] + ('' if _converged(best[1]) else ' [미수렴, loglike 최대 결과 사용]') + ('' if len(tried) == 1 else f' (시도 {len(tried)}회)')
    return best[1], best[2], dropped, X, note

**`coef_table`**: 외생변수 계수, p-value, 변화율(%)을 표로 만듭니다.

In [12]:
def coef_table(res, X):
    t = pd.DataFrame({'coef': res.params[X.columns], 'p': res.pvalues[X.columns]})
    t['pct_change'] = (np.exp(t.coef) - 1) * 100
    return t

**`cumulative`**: 변수별 누적 효과: lag0~3 계수의 합과 p-value (합 = 0 검정). X에 있는 ASOS·특보 변수 모두

In [13]:
def cumulative(res, X):
    """변수별 누적 효과: lag0~3 계수의 합과 p-value (합 = 0 검정). X에 있는 ASOS·특보 변수 모두"""
    names = list(res.param_names)
    rows = []
    for w in WEATHER + ALERTS:
        cols = [c for c in [w] + [f'{w}_lag{k}' for k in LAGS] if c in names]      # lag만 쓰는 버전은 lag1~3 합
        if not cols:
            continue
        R = np.zeros(len(names))
        for c in cols:
            R[names.index(c)] = 1.0
        t = res.t_test(R)
        eff, se, p = float(np.ravel(t.effect)[0]), float(np.ravel(t.sd)[0]), float(np.ravel(t.pvalue)[0])
        rows.append({'변수': w, 'cum_coef': eff, 'se': se, 'p': p, '변화율': (np.exp(eff) - 1) * 100})
    return pd.DataFrame(rows)

**`ljung_box`**: 잔차의 Ljung-Box 검정(lag 7, 14).

In [14]:
def ljung_box(res, order, seasonal_order):
    r = res.resid.iloc[2 * seasonal_order[3]:].dropna()        # 앞쪽 초기화 구간(14일)과 y가 NaN인 날 제외
    df = order[0] + order[2] + seasonal_order[0] + seasonal_order[2]
    return acorr_ljungbox(r, lags=[7, 14], model_df=df), len(r)

**`alert_window`**: any_alert=1인 날과 그다음 3일 (2024년 구간, 불리언 Series)

In [15]:
def alert_window(d, st):
    """any_alert=1인 날과 그다음 3일 (2024년 구간, 불리언 Series)"""
    a = d[d.station == st].set_index('date').sort_index().asfreq('D').any_alert.fillna(0)
    return ((a + a.shift(1) + a.shift(2) + a.shift(3)).fillna(0) > 0)[VALID_START:VALID_END]

**`validate`**: 계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).

In [16]:
def validate(res, y, X, total, window=None):
    """계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).
    window를 주면 그 날들(특보일+3일) 중 가리지 않은 날만으로도 계산"""
    full = res.apply(y, exog=X, refit=False)                    # 2014~2024 전체에 같은 계수 적용(2024 y는 전날까지만 예측에 쓰임)
    pred = np.exp(full.get_prediction(start=VALID_START, end=VALID_END, dynamic=False).predicted_mean)
    act = total[VALID_START:VALID_END]
    ok = y[VALID_START:VALID_END].notna()                      # 가리기된 날 제외
    err = act[ok] - pred[ok]
    naive = total.shift(7)[VALID_START:VALID_END][ok]            # 참고: 7일 전 같은 요일 값
    out = {'n_days': int(ok.sum()), 'n_masked': int((~ok).sum()),
           'RMSE': float(np.sqrt((err ** 2).mean())), 'MAPE': float((err.abs() / act[ok]).mean() * 100),
           'naive7_RMSE': float(np.sqrt(((act[ok] - naive) ** 2).mean())), 'naive7_MAPE': float(((act[ok] - naive).abs() / act[ok]).mean() * 100)}
    detail = pd.DataFrame({'actual': act, 'pred': pred, 'ok': ok})
    if window is not None:
        m = ok & window
        e = act[m] - pred[m]
        out.update(n_alert=int(m.sum()), RMSE_alert=float(np.sqrt((e ** 2).mean())), MAPE_alert=float((e.abs() / act[m]).mean() * 100))
        detail['window'] = window
    out['detail'] = detail
    return out

**`run_station`**: 한 역·한 모형을 적합하고 검증 지표까지 계산합니다.

In [17]:
def run_station(d, st, order, seasonal_order, season=False, retry=False, model='B', hint=None):
    y, X, grp, total = prepare(d, st, season, model)
    res, msgs, dropped, Xu, note = fit_station(y, X, order, seasonal_order, retry, hint)
    lb, nlb = ljung_box(res, order, seasonal_order)
    return dict(station=st, group=grp, order=order, seasonal_order=seasonal_order, res=res, msgs=msgs, dropped=dropped, X=Xu, lb=lb, nlb=nlb, note=note,
                valid=validate(res, y, Xu, total, alert_window(d, st)), y=y)

병렬 작업에서 데이터를 한 번만 읽어 두는 전역 변수.

In [18]:
_DATA = None

**`_task_full`**: 병렬 작업 하나: 한 역·한 모형의 지표, 계수, 누적 효과, 일별 예측을 반환합니다.

In [19]:
def _task_full(args):
    global _DATA
    st, order, seasonal, season, retry, tag, model, *rest = args
    if _DATA is None:
        _DATA = load()
    hint = None
    if rest and rest[0]:                                        # 다른 모형(rest[0], B 또는 B_lag)의 계수를 초기값으로 쓰는 재시도 (C 모형, 수유·연신내)
        hint = run_station(_DATA, st, order, seasonal, season, True, rest[0])['res'].params.to_dict()
    r = run_station(_DATA, st, order, seasonal, season, retry, model, hint)
    res, v, lb = r['res'], r['valid'], r['lb']
    lab = label(order, seasonal)
    coefs = coef_table(res, r['X']).reset_index().rename(columns={'index': '변수', 'pct_change': '변화율'})
    cum = cumulative(res, r['X'])
    for t in (coefs, cum):
        if tag:
            t.insert(0, '버전', tag)
        t.insert(0, '차수', lab); t.insert(0, '유형', r['group']); t.insert(0, '역', st)
    metrics = {'역': st, '유형': r['group'], '차수': lab, 'RMSE': v['RMSE'], 'MAPE': v['MAPE'], 'AIC': res.aic,
               'LjungBox_p_lag7': float(lb.lb_pvalue.iloc[0]), 'LjungBox_p_lag14': float(lb.lb_pvalue.iloc[1]),
               '수렴': bool(res.mle_retvals.get('converged')), '반복': int(res.mle_retvals.get('iterations') or 0), '경고수': len(r['msgs']),
               '평가일수': v['n_days'], 'naive7_MAPE': v['naive7_MAPE'], '뺀 열': ','.join(r['dropped'])}
    if tag:
        metrics = {**{'역': st, '유형': r['group'], '버전': tag}, **{k: m for k, m in metrics.items() if k not in ('역', '유형')}, '적합': r['note']}
    detail = v['detail'].reset_index().rename(columns={'index': 'date'})
    detail.insert(0, '모형', tag); detail.insert(0, '역', st)
    extra = {k: v[k] for k in ('n_alert', 'RMSE_alert', 'MAPE_alert')}
    return metrics, coefs, cum, detail, extra

**`_task`**: 병렬 작업 하나(`_task_full`에서 지표, 계수, 누적 효과만 돌려줌).

In [20]:
def _task(args):
    m, c, k, _, _ = _task_full(args)
    return m, c, k

**`stage3`**: B 모형(ASOS 날씨만)을 `summer_vacation` **없음 / 있음** 두 버전으로 9개 역에서 적합하고(현재 설정의 마스크, 미수렴이면 재시도), 날씨 누적 효과와 지표를 비교해 `results/sarimax_B_season_*.csv`로 저장합니다 (그림 7의 입력).

In [21]:
def stage3():
    """summer_vacation 없음/있음 비교 (확정 차수). 미수렴이면 초기값·maxiter를 바꿔 다시 시도"""
    RESULTS.mkdir(exist_ok=True)
    versions = [('없음', False), ('있음', True)]
    o, so = CANDIDATES[0]
    jobs = [(st, o, so, season, True, tag, 'B') for st in STATIONS for tag, season in versions]
    out = Parallel(n_jobs=WORKERS)(delayed(_task)(j) for j in jobs)             # 병렬 적합 (joblib)
    V = pd.DataFrame([m for m, _, _ in out])
    A = pd.concat([c for _, c, _ in out], ignore_index=True)
    K = pd.concat([k for _, _, k in out], ignore_index=True)
    V.to_csv(RESULTS / 'sarimax_B_season_compare.csv', index=False, encoding='utf-8-sig')
    A.to_csv(RESULTS / 'sarimax_B_season_coefs.csv', index=False, encoding='utf-8-sig')
    K.to_csv(RESULTS / 'sarimax_B_season_cumulative.csv', index=False, encoding='utf-8-sig')

    pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 60, 'display.float_format', lambda x: f'{x:,.4f}')
    order_st = [s_ for g in TYPE_ORDER for s_ in STATIONS if V[V.역 == s_].유형.iloc[0] == g]
    print(f'[summer_vacation 없음/있음 — {MAIN}, 2024 롤링 예측]')
    show = V.set_index(['역', '버전']).reindex(pd.MultiIndex.from_product([order_st, ['없음', '있음']], names=['역', '버전']))
    print(show[['유형', 'RMSE', 'MAPE', 'AIC', 'LjungBox_p_lag7', 'LjungBox_p_lag14', '수렴', '반복', '경고수', '적합']].to_string())
    print('\n[summer_vacation 계수]')
    sv = A[(A.버전 == '있음') & (A.변수 == 'summer_vacation')].set_index('역').reindex(order_st)
    print(sv[['유형', 'coef', 'p', '변화율']].to_string())

    # 유형별 요약
    rows = []
    for ver in ['없음', '있음']:
        for g in TYPE_ORDER:
            v = V[(V.버전 == ver) & (V.유형 == g)]
            row = {'버전': ver, '유형': g, '역수': len(v), 'MAPE평균': v.MAPE.mean(), 'AIC평균': v.AIC.mean(), 'RMSE평균': v.RMSE.mean()}
            for w in WEATHER:
                k = K[(K.버전 == ver) & (K.유형 == g) & (K.변수 == w)]
                row[f'{w}_누적평균'] = k.cum_coef.mean(); row[f'{w}_변화율%'] = (np.exp(k.cum_coef.mean()) - 1) * 100
                row[f'{w}_유의역수'] = int((k.p < 0.05).sum())
            rows.append(row)
    S = pd.DataFrame(rows)
    S.to_csv(RESULTS / 'sarimax_B_season_by_type.csv', index=False, encoding='utf-8-sig')
    print('\n[유형별 평균: MAPE, AIC와 날씨 누적 효과(lag0~3 합)]'); print(S.to_string(index=False))

    # 폭염 효과 유형 비교
    print('\n[폭염 누적 효과 (heat_excess lag0~3 합)  없음 → 있음]')
    H = K[K.변수 == 'heat_excess'].pivot(index='역', columns='버전', values='cum_coef').reindex(order_st)
    Hp = K[K.변수 == 'heat_excess'].pivot(index='역', columns='버전', values='p').reindex(order_st)
    H.insert(0, '유형', [V[V.역 == x].유형.iloc[0] for x in H.index])
    H['p(있음)'] = Hp['있음']
    print(H.to_string())
    hs = K[(K.버전 == '있음') & (K.변수 == 'heat_excess')]
    emp, oth = hs[hs.유형 == 'Employment'].cum_coef, hs[hs.유형 != 'Employment'].cum_coef
    print(f'\n있음 버전: Employment 평균 {emp.mean():+.4f} (범위 {emp.min():+.4f}~{emp.max():+.4f}) | 나머지 6개 역 평균 {oth.mean():+.4f} (범위 {oth.min():+.4f}~{oth.max():+.4f})')
    print('Employment가 나머지 역 전부보다 더 큰(더 음수) 역:', [x for x in emp.index.map(lambda i: hs.loc[i, "역"])] if False else
          [hs.loc[i, '역'] for i in emp.index if emp[i] < oth.min()])

**`stage5`**: N·A·B·B_lag·C·C_lag 6개 모형을 9개 역에서 적합하고, 2024 롤링 예측 오차(전체 평가일 / 특보일+3일), N 대비 개선, 누적 효과, 특보 유의성을 계산해 `results/sarimax_NABC_*.csv`로 저장합니다.

In [22]:
def stage5():
    """N(날씨 없음) · A · B · B_lag · C · C_lag: 확정 차수, summer_vacation 포함, 9개 역.
    2024 롤링 예측 오차를 (1) 전체 평가일, (2) any_alert=1인 날과 그다음 3일로 계산. 수유·연신내의 C는 B 계수를 초기값으로 재시도"""
    from scipy.stats import wilcoxon
    RESULTS.mkdir(exist_ok=True)
    o, so = CANDIDATES[0]
    models = ['N', 'A', 'B', 'B_lag', 'C', 'C_lag']
    base = {'C': 'B', 'C_lag': 'B_lag'}
    jobs = [(st, o, so, True, True, m, m, base.get(m) if st in RETRY_C_FROM_B else None) for st in STATIONS for m in models]
    jobs.sort(key=lambda j: j[7] is None)                       # 오래 걸리는 재시도 작업을 먼저 시작
    out = Parallel(n_jobs=WORKERS)(delayed(_task_full)(j) for j in jobs)       # 병렬 적합 (joblib)
    V = pd.DataFrame([{**m, **e} for m, _, _, _, e in out]).rename(columns={'버전': '모형', 'RMSE': 'RMSE_전체', 'MAPE': 'MAPE_전체', 'n_alert': '일수_특보', 'RMSE_alert': 'RMSE_특보', 'MAPE_alert': 'MAPE_특보', '평가일수': '일수_전체'})
    A = pd.concat([c for _, c, _, _, _ in out], ignore_index=True).rename(columns={'버전': '모형'})
    K = pd.concat([k for _, _, k, _, _ in out], ignore_index=True).rename(columns={'버전': '모형'})
    D = pd.concat([d for _, _, _, d, _ in out], ignore_index=True)
    V['_o'] = V.역.map({x: i for i, x in enumerate(STATIONS)}); V['_m'] = V.모형.map({x: i for i, x in enumerate(models)})
    V = V.sort_values(['_o', '_m']).drop(columns=['_o', '_m']).reset_index(drop=True)
    V.to_csv(RESULTS / 'sarimax_NABC_compare.csv', index=False, encoding='utf-8-sig')
    A.to_csv(RESULTS / 'sarimax_NABC_coefs.csv', index=False, encoding='utf-8-sig')
    K.to_csv(RESULTS / 'sarimax_NABC_cumulative.csv', index=False, encoding='utf-8-sig')
    D.to_csv(RESULTS / 'sarimax_NABC_pred2024.csv', index=False, encoding='utf-8-sig')

    pd.set_option('display.width', 280, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.float_format', lambda x: f'{x:,.4f}')
    order_st = [x for g in TYPE_ORDER for x in STATIONS if V[V.역 == x].유형.iloc[0] == g]
    print(f'[모형별 지표 — {MAIN}, summer_vacation 포함, 2024 롤링 예측. 전체 평가일 / 특보일+3일]')
    show = V.set_index(['역', '모형']).reindex(pd.MultiIndex.from_product([order_st, models], names=['역', '모형']))
    print(show[['유형', 'MAPE_전체', 'MAPE_특보', 'RMSE_전체', 'RMSE_특보', 'AIC', '수렴', '반복', '일수_전체', '일수_특보']].to_string())
    print('\n[수렴 / 적합 방법 메모 (기본이 아닌 것)]')
    print(V[V.적합.ne('기본(lbfgs, maxiter=1000)') | ~V.수렴][['역', '모형', '수렴', '반복', '적합']].to_string(index=False))

    M = V.groupby(['유형', '모형']).agg(MAPE_전체=('MAPE_전체', 'mean'), MAPE_특보=('MAPE_특보', 'mean'), RMSE_전체=('RMSE_전체', 'mean'), RMSE_특보=('RMSE_특보', 'mean'),
                                       AIC평균=('AIC', 'mean'), 수렴역수=('수렴', 'sum')).reindex(pd.MultiIndex.from_product([TYPE_ORDER, models], names=['유형', '모형']))
    M.to_csv(RESULTS / 'sarimax_NABC_by_type.csv', encoding='utf-8-sig')
    print('\n[유형별 평균 지표]'); print(M.to_string())

    # N 대비 개선 (역별 + 유형 평균), 두 평가 구간
    rows = []
    for st in order_st:
        for part, (mc, rc, sub) in {'전체': ('MAPE_전체', 'RMSE_전체', 'ok'), '특보일+3일': ('MAPE_특보', 'RMSE_특보', 'window')}.items():
            dn = D[(D.역 == st) & (D.모형 == 'N')].set_index('date')
            sel = dn.ok if sub == 'ok' else (dn.ok & dn.window)
            ape_n = ((dn.actual - dn.pred).abs() / dn.actual)[sel]
            vn = V[(V.역 == st) & (V.모형 == 'N')].iloc[0]
            for m in models[1:]:
                dm = D[(D.역 == st) & (D.모형 == m)].set_index('date')
                ape_m = ((dm.actual - dm.pred).abs() / dm.actual)[sel]
                vm = V[(V.역 == st) & (V.모형 == m)].iloc[0]
                diff = (ape_n - ape_m).dropna()
                try:
                    pw = float(wilcoxon(diff).pvalue) if (diff != 0).any() else np.nan
                except ValueError:
                    pw = np.nan
                rows.append({'역': st, '유형': vn.유형, '모형': m, '구간': part, '일수': int(sel.sum()), 'MAPE_N': vn[mc], 'MAPE_모형': vm[mc],
                             'MAPE_개선%p': vn[mc] - vm[mc], 'MAPE_개선%': (vn[mc] - vm[mc]) / vn[mc] * 100,
                             'RMSE_N': vn[rc], 'RMSE_모형': vm[rc], 'RMSE_개선%': (vn[rc] - vm[rc]) / vn[rc] * 100, 'wilcoxon_p': pw})
    R = pd.DataFrame(rows)
    T = R.groupby(['유형', '모형', '구간']).agg(일수=('일수', 'mean'), MAPE_N=('MAPE_N', 'mean'), MAPE_모형=('MAPE_모형', 'mean'), **{'MAPE_개선%p': ('MAPE_개선%p', 'mean'), 'MAPE_개선%': ('MAPE_개선%', 'mean'),
                                                                                  'RMSE_개선%': ('RMSE_개선%', 'mean'), '모형이_나은_역수': ('MAPE_개선%p', lambda v: int((v > 0).sum())),
                                                                                  'p<0.05_역수': ('wilcoxon_p', lambda v: int((v < 0.05).sum()))}).reset_index()
    T.insert(0, '역', '(유형 평균)')
    allr = pd.concat([R, T], ignore_index=True)
    allr.to_csv(RESULTS / 'sarimax_NABC_vs_N.csv', index=False, encoding='utf-8-sig')
    print('\n[N 대비 개선 — 유형별 평균 (MAPE_개선%p = N의 MAPE − 모형의 MAPE, 양수면 모형이 나음)]')
    TT = T.set_index(['구간', '유형', '모형']).reindex(pd.MultiIndex.from_product([['전체', '특보일+3일'], TYPE_ORDER, models[1:]], names=['구간', '유형', '모형']))
    print(TT[['일수', 'MAPE_N', 'MAPE_모형', 'MAPE_개선%p', 'MAPE_개선%', 'RMSE_개선%', '모형이_나은_역수', 'p<0.05_역수']].to_string())
    for m in ['B', 'B_lag']:
        print(f'\n[역별: {m}가 N보다 얼마나 좋아지나 — 특보일+3일 / 전체]')
        X = R[R.모형 == m].pivot(index='역', columns='구간', values=['일수', 'MAPE_N', 'MAPE_모형', 'MAPE_개선%p', 'MAPE_개선%', 'wilcoxon_p']).reindex(order_st)
        print(X.to_string())

    # 특보 누적 효과 (C, C_lag, A) 유의 역 수와 수유·연신내 재시도 결과
    G = K[K.변수.isin(ALERTS)].groupby(['변수', '모형']).agg(유의_역수=('p', lambda v: int((v < 0.05).sum())), 음수_역수=('cum_coef', lambda v: int((v < 0).sum()))).reindex(
        pd.MultiIndex.from_product([ALERTS, ['A', 'C', 'C_lag']], names=['변수', '모형']))
    G.to_csv(RESULTS / 'sarimax_NABC_alert_significance.csv', encoding='utf-8-sig')
    print('\n[특보 누적 효과 p<0.05인 역 수 /9]'); print(G.to_string())
    print('\n[수유·연신내 C 재시도 후: 특보 누적 효과]')
    print(K[K.역.isin(RETRY_C_FROM_B) & K.모형.isin(['C', 'C_lag']) & K.변수.isin(ALERTS)][['역', '모형', '변수', 'cum_coef', 'p']].to_string(index=False))

**실행**: 6개 모형 × 9개 역 = 54개 적합을 병렬로 돌리고 결과 표를 저장합니다 (약 5분). 결과는 `results/`에 저장됩니다 (이미 같은 이름의 파일이 있으면 덮어씀).

In [23]:
stage5()

[모형별 지표 — (1,0,1)(1,1,1,7), summer_vacation 포함, 2024 롤링 예측. 전체 평가일 / 특보일+3일]
                               유형  MAPE_전체  MAPE_특보    RMSE_전체    RMSE_특보          AIC    수렴   반복  일수_전체  일수_특보
역     모형                                                                                                       
시청    N                Employment   5.2136   4.6916 7,402.3046 5,255.5167  -5,324.0636  True   66    339    104
      A                Employment   5.0625   4.4620 7,279.9468 4,904.5059  -5,331.0360  True  125    339    104
      B                Employment   5.0115   4.4993 7,225.6452 4,742.9664  -5,384.7457  True  241    339    104
      B_lag            Employment   5.1410   4.6221 7,308.8841 4,865.6837  -5,344.4900  True  313    339    104
      C                Employment   5.0987   4.7449 7,454.0027 5,635.3296  -5,375.3114  True  337    339    104
      C_lag            Employment   5.2341   4.9092 7,551.3709 5,836.8569  -5,343.7875  True  290    339    104
역삼    N                Empl

**summer_vacation 있음/없음 비교 실행**: B 모형(ASOS 날씨만)을 `summer_vacation` 없음/있음으로 9개 역에서 적합합니다 (18개 적합, 약 3분). 휴가철 더미를 넣으면 폭염 누적 효과가 얼마나 줄어드는지 보는 비교이며, 결과는 `results/sarimax_B_season_*.csv`에 저장됩니다 (이미 있으면 덮어씀).

In [24]:
stage3()

[summer_vacation 없음/있음 — (1,0,1)(1,1,1,7), 2024 롤링 예측]
                            유형       RMSE   MAPE          AIC  LjungBox_p_lag7  LjungBox_p_lag14    수렴   반복  경고수                                 적합
역     버전                                                                                                                                           
시청    없음            Employment 7,235.9918 5.0358  -5,385.8899           0.0014            0.0001  True  246    0            기본(lbfgs, maxiter=1000)
      있음            Employment 7,225.6452 5.0115  -5,384.7457           0.0014            0.0001  True  241    0            기본(lbfgs, maxiter=1000)
역삼    없음            Employment 3,721.9155 2.5859  -8,581.0141           0.0000            0.0000  True  286    0            기본(lbfgs, maxiter=1000)
      있음            Employment 3,716.6521 2.5702  -8,584.3982           0.0000            0.0000  True  420    0            기본(lbfgs, maxiter=1000)
선릉    없음            Employment 3,267.2209 2.4839  -8,825.